# Zenith AI — Google Colab Training

Trains the ~100.7M-parameter `configs/zenith_colab.yaml` variant of Zenith on a free Colab GPU.

**Before running:** `Runtime` menu → `Change runtime type` → set Hardware accelerator to **GPU** (T4 is the free-tier default).

**Unlike the Kaggle version of this notebook, this one has to be run manually** — Colab's free tier has no API for launching/monitoring a run headlessly the way `kaggle kernels push` does. Open this in your browser, `Runtime` → `Run all`, and leave the tab open (or check back periodically; Colab disconnects idle tabs and has its own multi-hour session cap).

**Persistence via Google Drive**: Colab's local disk (`/content`) is wiped every session. This notebook mounts your Drive and points both the packed dataset and `checkpoints/` at `/content/drive/MyDrive/zenith_ai/` — so re-running this notebook in a fresh session automatically resumes from `latest.pt` and skips re-downloading/re-tokenizing data, no manual dataset-reattach step needed (that was a Kaggle-specific workaround, not needed here).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/zenith_ai"
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print("Drive mounted, persistent dir ready:", DRIVE_ROOT)

In [ ]:
REPO = "/content/Zenith-AI"

!rm -rf {REPO}
!git clone --depth 1 https://github.com/ItzAditya43/Zenith-AI.git {REPO}

## Make sure PyTorch supports whatever GPU Colab assigned

Same defensive check used in the Kaggle notebook: free-tier accelerators can occasionally be an older architecture (e.g. a Tesla K80/P100 on some Colab allocations) that the preinstalled PyTorch build no longer supports, even though `torch.cuda.is_available()` still reports `True`. This reinstalls a compatible build if needed and does a real matmul smoke test rather than trusting `is_available()` alone.

In [ ]:
import subprocess

def gpu_compute_capability():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip().splitlines()
    return out[0] if out else None

cc = gpu_compute_capability()
print("GPU compute capability:", cc)

if cc is not None and float(cc) < 7.0:
    print("Older GPU architecture detected — reinstalling a compatible PyTorch build (cu118)...")
    !pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu118
else:
    print("Stock PyTorch build should support this GPU, skipping reinstall.")

In [ ]:
!pip install -q tokenizers datasets pyyaml tqdm

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    x = torch.randn(4, 4, device="cuda") @ torch.randn(4, 4, device="cuda")
    torch.cuda.synchronize()
    print("CUDA matmul smoke test OK:", x.shape)
else:
    raise RuntimeError("No GPU available — set Runtime > Change runtime type > GPU and rerun.")

## Data pipeline (persisted to Drive)

Downloads TinyStories, trains an 8192-vocab BPE tokenizer from scratch, then tokenizes and packs it — all written straight to `DRIVE_ROOT` so a future session skips this entirely if the files already exist.

In [ ]:
if not os.path.exists(f"{DRIVE_ROOT}/data/train.txt"):
    !cd {REPO} && python -m zenith.data.prepare dump --output-dir {DRIVE_ROOT}/data
else:
    print("Found existing corpus on Drive, skipping download.")

In [ ]:
if not os.path.exists(f"{DRIVE_ROOT}/data/tokenizer.json"):
    !cd {REPO} && python -m zenith.tokenizer.train_tokenizer --input {DRIVE_ROOT}/data/train.txt --output {DRIVE_ROOT}/data/tokenizer.json --vocab-size 8192
else:
    print("Found existing tokenizer on Drive, skipping.")

In [ ]:
if not os.path.exists(f"{DRIVE_ROOT}/data/train_packed.npy"):
    !cd {REPO} && python -m zenith.data.prepare pack --input {DRIVE_ROOT}/data/train.txt --tokenizer {DRIVE_ROOT}/data/tokenizer.json --output {DRIVE_ROOT}/data/train_packed.npy --seq-len 512
if not os.path.exists(f"{DRIVE_ROOT}/data/val_packed.npy"):
    !cd {REPO} && python -m zenith.data.prepare pack --input {DRIVE_ROOT}/data/val.txt --tokenizer {DRIVE_ROOT}/data/tokenizer.json --output {DRIVE_ROOT}/data/val_packed.npy --seq-len 512
else:
    print("Found packed shards on Drive, skipping.")

## Train

~100.7M params. Reads/writes everything under `configs/zenith_colab.yaml`'s Drive paths, so this cell is safe to interrupt and rerun — it auto-resumes from `checkpoints/latest.pt` on Drive. Watch the first few hundred steps' `tok/s` figure and compare against the `total_steps` budget in the config; if this GPU is slower than what the config assumed, lower `total_steps` and rerun rather than letting it run indefinitely past Colab's own session cap.

In [ ]:
!cd {REPO} && python -m zenith.training.train configs/zenith_colab.yaml

## Try it

In [ ]:
import sys
sys.path.insert(0, REPO)
os.chdir(REPO)

from zenith.cli import load_model_and_tokenizer
from zenith.inference.generate import generate_text

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tok, cfg = load_model_and_tokenizer(f"{DRIVE_ROOT}/checkpoints/latest.pt", f"{DRIVE_ROOT}/data/tokenizer.json", device)

for prompt in ["Once upon a time", "The little dog was very", "Tom and Lily went to the park"]:
    text = generate_text(model, tok, prompt, max_new_tokens=100, temperature=0.7, top_k=40, device=device)
    print("PROMPT:", prompt)
    print("OUTPUT:", prompt + text)
    print()